# Noonchi Translator — mBART Fine-tuning

Fine-tunes `facebook/mbart-large-50-many-to-many-mmt` on the formality-labeled EN–KR corpus.

**Before running:** Set Runtime → Change runtime type → GPU (T4 or better).

**Training time estimates:**
- 50K rows, 3 epochs, batch 4 × grad_accum 8 on T4: ~5–8 hours
- Full 423K rows: use A100 (Colab Pro) or a university cluster

All checkpoints are saved to Google Drive so a session disconnect doesn't lose progress.

In [ ]:
# Cell 1 — Verify GPU is assigned before doing anything else
!nvidia-smi

In [ ]:
# Cell 2 — Mount Google Drive
# All model checkpoints write here so disconnects don't lose progress.
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/noonchi', exist_ok=True)
os.makedirs('/content/drive/MyDrive/noonchi/checkpoints', exist_ok=True)

In [ ]:
# Cell 3 — Install dependencies
# transformers 4.41+ required for eval_strategy rename and text_target tokenizer API.
# accelerate is a hard requirement of HuggingFace Trainer (often missing from requirements).
# mecab + konlpy are needed for formality_accuracy evaluation.
!pip install -q 'transformers>=4.41.0' 'accelerate>=0.26.0' datasets sentencepiece sacrebleu
!apt-get install -y -q mecab libmecab-dev mecab-ipadic-utf8
!pip install -q mecab-python3 konlpy

In [ ]:
# Cell 4 — Clone repo and set up Python path
# If your repo is private, authenticate first:
#   from google.colab import userdata
#   token = userdata.get('GITHUB_TOKEN')
#   !git clone https://{token}@github.com/cyatco/noonchi-translator.git /content/noonchi-translator
!git clone https://github.com/cyatco/noonchi-translator.git /content/noonchi-translator

import sys
sys.path.insert(0, '/content/noonchi-translator')
%cd /content/noonchi-translator

# Verify imports work
from backend.model.train import load_model_and_tokenizer
from backend.model.dataset import load_split
print('Imports OK')

In [ ]:
# Cell 5 — Copy training data from Drive
# Upload train.tsv and val.tsv to /content/drive/MyDrive/noonchi/ first.
# These files are too large to commit to git (~50MB each).
!cp /content/drive/MyDrive/noonchi/train.tsv data/train.tsv
!cp /content/drive/MyDrive/noonchi/val.tsv data/val.tsv
!cp /content/drive/MyDrive/noonchi/test.tsv data/test.tsv

# Verify line counts
!wc -l data/train.tsv data/val.tsv data/test.tsv

In [ ]:
# Cell 6 — Smoke test: load model and one batch before committing to full training
# If this fails, debug here rather than 30 minutes into a training run.
model, tokenizer = load_model_and_tokenizer()
ds = load_split('data/val.tsv', tokenizer)
sample = ds[0]
print('Dataset size:', len(ds))
print('Sample keys:', list(sample.keys()))
print('input_ids length:', len(sample['input_ids']))
print('labels length:', len(sample['labels']))
print('decoder_start_token_id:', model.config.decoder_start_token_id)
print('forced_bos_token_id:', model.config.forced_bos_token_id)
del model  # free VRAM before training

In [ ]:
# Cell 7 — Train
# --max-rows 50000: stratified 50K sample fits a free T4 session (~5–8 hours, 3 epochs)
# Remove --max-rows to train on the full 423K dataset (needs A100 or ~40+ hours)
!python -m backend.model.train \
    --data data/train.tsv \
    --output /content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart \
    --max-rows 50000

In [ ]:
# Cell 8 — Evaluate on held-out test split
# Sanity bars (50K model): chrF > 20, formality_accuracy > 0.60
# Target bars (full model): chrF > 30, formality_accuracy > 0.80
!python -m backend.model.evaluate \
    --model /content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart \
    --test data/test.tsv \
    --batch-size 32